##### LSTM Network - Prediction v9 (return-first multi-task model)\n\nThis notebook follows the same project workflow as the existing v7 example. The notebook remains intentionally thin: `StockPrediction`, `StockData`, `LongShortTermMemory`, `Plotter`, `train_LSTM_network` and `InferenceRunner` contain the implementation. v9 changes the target to next-session log returns and trains shared direction, expected-return and q10/q50/q90 heads.\n

In [ ]:
%%capture
%pip install -r requirements.txt

In [ ]:
# Copyright 2020-2026 Jordi Corbilla. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

import os
import warnings
import secrets
import pandas as pd
import argparse
import numpy as np
import pickle

from stock_prediction_class import StockPrediction
from stock_prediction_lstm import LongShortTermMemory
from stock_prediction_numpy import StockData
from stock_prediction_plotter import Plotter
from stock_prediction_readme_generator import ReadmeGenerator
from stock_prediction_deep_learning import train_LSTM_network

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # or '3' to suppress all messages

# Suppress other warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow")
warnings.filterwarnings("ignore", message=".*np.object.*", category=FutureWarning)

import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import timedelta, datetime
from pandas.tseries.offsets import BDay

os.environ["PATH"] += os.pathsep + 'C:/Program Files (x86)/Graphviz2.38/bin/'

In [ ]:
STOCK_TICKER = "^FTSE"
STOCK_START_DATE = pd.to_datetime('2017-01-01')
end_date = datetime.today()
duration = end_date - STOCK_START_DATE
STOCK_VALIDATION_DATE = STOCK_START_DATE + 0.8 * duration

MODEL_VERSION = 'v9'
EPOCHS = 20
BATCH_SIZE = 10
TIME_STEPS = 60
USE_RETURNS = True
VALIDATION_FRACTION = 0.15
SEED = 42
FORECAST_DAYS = 1

TODAY_RUN = datetime.today().strftime("%Y%m%d")
TOKEN = STOCK_TICKER + '_v9_' + TODAY_RUN + '_' + secrets.token_hex(8)
GITHUB_URL = "https://github.com/JordiCorbilla/stock-prediction-deep-neural-learning/raw/master/"

print('Ticker: ' + STOCK_TICKER)
print('Start Date: ' + STOCK_START_DATE.strftime("%Y-%m-%d"))
print('Validation Date: ' + STOCK_VALIDATION_DATE.strftime("%Y-%m-%d"))
print('Model Version: ' + MODEL_VERSION)
print('Target: next-session log return')
print('Test Run Folder: ' + TOKEN)

PROJECT_FOLDER = os.path.join(os.getcwd(), TOKEN)
if not os.path.exists(PROJECT_FOLDER):
    os.makedirs(PROJECT_FOLDER)

stock_prediction = StockPrediction(
    STOCK_TICKER,
    STOCK_START_DATE,
    STOCK_VALIDATION_DATE,
    PROJECT_FOLDER,
    GITHUB_URL,
    EPOCHS,
    TIME_STEPS,
    TOKEN,
    BATCH_SIZE,
)

train_LSTM_network(
    stock_prediction,
    use_returns=USE_RETURNS,
    model_version=MODEL_VERSION,
    forecast_horizon=FORECAST_DAYS,
    trend_window=60,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
)


## Infer the Future Data (Predictions)\n\nThe same `InferenceRunner` used by the existing project loads the saved v9 model and recursively forecasts future log returns, which are converted back into prices for the familiar future-price chart.\n

In [ ]:
DIRECTION_THRESHOLD = 0.55
MAG_CLIP_PCT = 90
BLEND_ALPHA = 1.0

STOCHASTIC_PATHS = 50
STOCHASTIC_SEED = 42
STOCHASTIC_SIGMA_MULT = 0.6
STOCHASTIC_LOOKBACK = 120

from stock_prediction_deep_learning_inference import InferenceRunner

FORECAST_DAYS = 30
USE_BUSINESS_DAYS = True
PLOT_HISTORY_DAYS = 200
USE_RETURNS = True
USE_DELTAS = False
CLIP_NEGATIVE = True
CONFORMAL_COVERAGE = 0.90
EXCHANGE_CALENDAR = "XLON"

runner = InferenceRunner(
    run_folder=PROJECT_FOLDER,
    ticker=STOCK_TICKER,
    start_date=STOCK_START_DATE,
    validation_date=STOCK_VALIDATION_DATE,
    github_url=GITHUB_URL,
    epochs=EPOCHS,
    time_steps=TIME_STEPS,
    token=TOKEN,
    batch_size=BATCH_SIZE,
    forecast_days=FORECAST_DAYS,
    use_business_days=USE_BUSINESS_DAYS,
    plot_history_days=PLOT_HISTORY_DAYS,
    use_returns=USE_RETURNS,
    use_deltas=USE_DELTAS,
    clip_negative=CLIP_NEGATIVE,
    blend_alpha=BLEND_ALPHA,
    direction_threshold=DIRECTION_THRESHOLD,
    mag_clip_pct=MAG_CLIP_PCT,
    stochastic_paths=STOCHASTIC_PATHS,
    stochastic_seed=STOCHASTIC_SEED,
    stochastic_sigma_mult=STOCHASTIC_SIGMA_MULT,
    stochastic_lookback=STOCHASTIC_LOOKBACK,
    conformal_coverage=CONFORMAL_COVERAGE,
    exchange_calendar=EXCHANGE_CALENDAR,
)
runner.run()


In [ ]:
metrics_path = os.path.join(PROJECT_FOLDER, 'return_forecast_metrics.json')
forecast_path = os.path.join(PROJECT_FOLDER, 'forecast_metrics.json')
future_path = os.path.join(PROJECT_FOLDER, 'future_predictions.csv')

with open(metrics_path, encoding='utf-8') as handle:
    return_metrics = json.load(handle)

with open(forecast_path, encoding='utf-8') as handle:
    price_metrics = json.load(handle)

print('v9 held-out return metrics')
display(pd.DataFrame([return_metrics]).round(6))

print('Held-out price reconstruction metrics')
display(pd.DataFrame([price_metrics]).round(6))

print('First five future predictions')
display(pd.read_csv(future_path).head())
